In [ ]:
from analyses import parse
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import polars as pl
import numpy as np
import colorir as cl
from pathlib import Path
from tqdm import tqdm

In [ ]:
sim_paths = {}
for path in Path("../runs/speed/").iterdir():
    sim_type = path.name
    gamma = 20 - int(sim_type.split("-")[1])
    for replica_path in path.iterdir():
        replica = int(replica_path.name)
        sim_paths[(gamma, replica)] = replica_path
sims = parse.parse_cells_multiple(sim_paths.values(), 30)
sim_ids = list(sim_paths.keys())
for i in range(len(sims)):
    sims[i] = sims[i].with_columns(
        gamma=sim_ids[i][0], 
        replica=sim_ids[i][1]
    )
cellveldf = pl\
    .concat(sims)\
    .with_columns(
        displ=(pl.col("center_x") ** 2 + pl.col("center_y") ** 2).sqrt()
    )
cellveldf

In [ ]:
grouppers = ["gamma", "replica", "wtime"]
displdf = cellveldf.group_by(grouppers).agg(
    cluster_x=pl.col("center_x").mean(),
    cluster_y=pl.col("center_y").mean(),
    mean_displ=pl.col("displ").mean(),
    med_displ=pl.col("displ").median()
).sort(grouppers)
displdf

In [ ]:
max_time = displdf["wtime"].max()
stdf = displdf\
    .group_by(["gamma"])\
    .agg(
        stime=pl.col("wtime")\
            .filter(
                pl.col("mean_displ") <= pl.col("mean_displ").filter(pl.col("wtime") > 1e7).mean()
            )\
            .min(),
        ptime=pl.col("wtime")\
            .filter(
                pl.col("mean_displ") <= 180
            )\
            .min().fill_null(25e4 + 10e4)
    )
stdf

In [ ]:
# These sims should contain a single cell!!
sim_paths = {}
for sp in Path("../runs/msd/").iterdir():
    sim_type = sp.name
    gamma = 20 - int(sim_type.split("-")[1])
    for p in sp.iterdir():
        replica = int(p.name)
        sim_paths[(gamma, replica)] = p
    
sims = parse.parse_cells_multiple(
    sim_paths.values(),
    n_workers=24
)
celldf = pl.concat([sim.with_columns(gamma=k[0], replica=k[1]) for k, sim in zip(sim_paths, sims)])
assert (celldf["index"] == 0).all()
celldf

In [ ]:
cuts = pl.DataFrame({
    "dt": np.linspace(
        0,
        5e6,
        21
    )
})
    
trajdf = celldf\
    .filter(
        pl.col("time") % 100 == 0,
        gamma=0
    )\
    .select(
        "time",
        "center_x",
        "center_y",
        "replica"
    )\
    .sort(["replica", "time"])\
    .join(cuts, how="cross")\
    .filter(pl.col("time") <= pl.col("dt"))
trajdf

In [ ]:
px.line(
    trajdf, 
    x="center_x", 
    y="center_y",
    color="replica",
    animation_frame="dt"
).update_traces(
    visible='legendonly'
).update_layout(
    xaxis_range=[0, 500], 
    yaxis_range=[0, 500],
    width=500, 
    height=500
)

In [ ]:
filterdf = celldf.select(
    "gamma",
    "replica",
    "time",
    "center_x",
    "center_y",
    displ=(pl.col("center_x") ** 2 + pl.col("center_y") ** 2) ** 0.5
).filter(
    # Optional gamma filtering
    pl.col("gamma").is_in([0, 12]),
    pl.col("time") >= 25e4,
).join(
    stdf, 
    on="gamma"
).filter(
    # Using ptime, change to stime if needed
    pl.col("time") <= pl.col("ptime"),
).sort([
    "gamma", 
    "replica",
    "time"
])
filterdf

In [ ]:
interval = 1000_000
dts = np.concat([
    [10], 
    np.arange(
        interval,
        2_000_000 + interval,
        interval,
        dtype=int
    )
])
# All numbers must be divisible by 10
assert np.equal(np.mod(dts / 10, 1), 0).all()
dts

In [ ]:
msddfs = []
groupped = filterdf.group_by(["gamma", "replica"])
for dt in tqdm(dts, mininterval=2):
    msd = groupped\
        .agg(
            len=pl.col("center_x").diff(n=dt / 10).drop_nulls().len(),
            msd=(
                pl.col("center_x").diff(n=dt / 10) ** 2
                + pl.col("center_y").diff(n=dt / 10) ** 2
            ).mean()
        )\
        .with_columns(dt=dt).filter(pl.col("len") >= 10)
    msddfs.append(msd)
msddf = pl.concat(msddfs)

In [ ]:
aggdf = msddf.group_by(["gamma", "dt"]).agg(
    mean=pl.col("msd").mean(),
    med=pl.col("msd").median(),
    min=pl.col("msd").min(),
    max=pl.col("msd").max(),
    std1=pl.col("msd").mean() + 1.5 * pl.col("msd").std(),
    std2=pl.col("msd").mean() - 1.5 * pl.col("msd").std()
).sort("dt").with_columns(
    pl.exclude(["gamma", "replica"]).log().name.prefix("ln_")
)
aggdf

In [ ]:
palette = cl.StackPalette().load("carnival")
grad = cl.PolarGrad(palette, domain=[0, 20])
palette

In [ ]:
fig = go.Figure()
for gamma, group in aggdf.group_by("gamma"):
    gamma = gamma[0]
    color = grad(gamma)
    fig.add_traces([
        go.Scatter(
            x=group["dt"],
            y=group["mean"],
            mode="lines",
            line_color=color,
            name=f"gamma-{gamma}",
            legendgroup=gamma
        ),
        go.Scatter(
            x=pl.concat([group["dt"], group["dt"][::-1]]),
            y=pl.concat([group["min"], group["max"][::-1]]),
            mode="lines",
            line_color="rgba(0, 0, 0, 0)",
            fillcolor=color,
            opacity=0.3,
            fill="toself",
            hoverinfo="skip",
            showlegend=False,
            legendgroup=gamma
        )
    ])
fig.update_layout(
    template="plotly_white",
    width=400,
    height=600
)

In [ ]:
fig2 = go.Figure(fig.update_layout(
    xaxis_type="log",
    yaxis_type="log",
    width=500,
    height=500
))
fig3 = go.Figure()
fig4 = go.Figure()
for gamma, group in aggdf.group_by("gamma"):
    gamma = gamma[0]
    dt = group["dt"]
    log_dt = dt.log()
    mean = group["mean"]
    poly = np.polynomial.Polynomial.fit(x=dt.log(), y=mean.log(), deg=5).convert()
    fig2.add_trace(go.Scatter(
        x=dt,
        y=np.exp(poly(dt.log())),
        mode="lines",
        line_dash="dash",
        line_color="#000000",
        showlegend=False,
        legendgroup=gamma
    ))

    fig3.add_trace(go.Scatter(
        x=np.exp(log_dt),
        y=poly(log_dt),
        mode="lines",
        line_color=grad(gamma),
        name=gamma,
        legendgroup=gamma
    ))
    
    first_deriv = poly.deriv()
    fig4.add_trace(go.Scatter(
        x=np.exp(log_dt),
        y=first_deriv(log_dt),
        mode="lines",
        line_color=grad(gamma),
        name=str(gamma)
    ))
fig2.show()
fig3.update_layout(
    template="plotly_white",
    xaxis_type="log",
    width=500,
    height=500
).show()
fig4.update_layout(
    template="plotly_white",
    xaxis_type="log"
)